# Video Game Sales Analysis - Predictive Modeling

This notebook implements and compares multiple machine learning models for predicting video game global sales.

**Models:**
1. Linear Regression
2. Random Forest
3. Support Vector Regression (SVR)
4. XGBoost

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

In [2]:
# Load dataset
df = pd.read_csv('../data/Video_Games.csv')
print(f'Dataset shape: {df.shape}')

Dataset shape: (16719, 16)


## Data Preparation

In [3]:
# Select features and target
features = ['Genre', 'Platform', 'Publisher', 'Developer']
target = 'Global_Sales'

# Create modeling dataset
model_df = df[features + [target]].copy()
model_df = model_df.dropna()

# Limit size for computational efficiency
if len(model_df) > 5000:
    model_df = model_df.sample(n=5000, random_state=42)

print(f'Modeling dataset shape: {model_df.shape}')
print(f'Features: {features}')
print(f'Target: {target}')

Modeling dataset shape: (5000, 5)
Features: ['Genre', 'Platform', 'Publisher', 'Developer']
Target: Global_Sales


In [4]:
# Split features and target
X = model_df[features]
y = model_df[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size: {X_test.shape[0]}')

Train size: 3500
Test size: 1500


In [5]:
# Preprocessing pipeline
categorical_features = features

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# Fit and transform
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(f'Encoded train shape: {X_train_encoded.shape}')
print(f'Encoded test shape: {X_test_encoded.shape}')

Encoded train shape: (3500, 1340)
Encoded test shape: (1500, 1340)


## 1. Linear Regression

In [6]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_encoded, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train_encoded)
lr_test_pred = lr_model.predict(X_test_encoded)

# Metrics
lr_train_rmse = np.sqrt(mean_squared_error(y_train, lr_train_pred))
lr_test_rmse = np.sqrt(mean_squared_error(y_test, lr_test_pred))
lr_train_mae = mean_absolute_error(y_train, lr_train_pred)
lr_test_mae = mean_absolute_error(y_test, lr_test_pred)
lr_train_r2 = r2_score(y_train, lr_train_pred)
lr_test_r2 = r2_score(y_test, lr_test_pred)

print('=== Linear Regression ===')
print(f'Train RMSE: {lr_train_rmse:.4f}')
print(f'Test RMSE:  {lr_test_rmse:.4f}')
print(f'Train MAE:  {lr_train_mae:.4f}')
print(f'Test MAE:   {lr_test_mae:.4f}')
print(f'Train R²:   {lr_train_r2:.4f}')
print(f'Test R²:    {lr_test_r2:.4f}')

=== Linear Regression ===
Train RMSE: 1.6198
Test RMSE:  1.1296
Train MAE:  0.4939
Test MAE:   0.6686
Train R²:   0.3343
Test R²:    0.1744


In [7]:
# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_rmse = np.sqrt(-cross_val_score(LinearRegression(), X_train_encoded, y_train, 
                                       cv=cv, scoring='neg_mean_squared_error').mean())
lr_cv_r2 = cross_val_score(LinearRegression(), X_train_encoded, y_train, 
                            cv=cv, scoring='r2').mean()

print(f'CV RMSE: {lr_cv_rmse:.4f}')
print(f'CV R²:   {lr_cv_r2:.4f}')

CV RMSE: 1.8931
CV R²:   -0.0064


## 2. Random Forest

In [8]:
# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_encoded, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train_encoded)
rf_test_pred = rf_model.predict(X_test_encoded)

# Metrics
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)

print('=== Random Forest ===')
print(f'Train RMSE: {rf_train_rmse:.4f}')
print(f'Test RMSE:  {rf_test_rmse:.4f}')
print(f'Train MAE:  {rf_train_mae:.4f}')
print(f'Test MAE:   {rf_test_mae:.4f}')
print(f'Train R²:   {rf_train_r2:.4f}')
print(f'Test R²:    {rf_test_r2:.4f}')

=== Random Forest ===
Train RMSE: 0.8500
Test RMSE:  1.4139
Train MAE:  0.2559
Test MAE:   0.5177
Train R²:   0.8167
Test R²:    -0.2934


In [9]:
# Cross-validation
rf_cv_rmse = np.sqrt(-cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                                       X_train_encoded, y_train, cv=cv, scoring='neg_mean_squared_error').mean())
rf_cv_r2 = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                            X_train_encoded, y_train, cv=cv, scoring='r2').mean()

print(f'CV RMSE: {rf_cv_rmse:.4f}')
print(f'CV R²:   {rf_cv_r2:.4f}')

CV RMSE: 2.0429
CV R²:   -0.1956


In [10]:
# Feature importance
feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(len(importance_df)), importance_df['Importance'])
plt.yticks(range(len(importance_df)), importance_df['Feature'])
plt.xlabel('Importance')
plt.title('Random Forest - Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Support Vector Regression (SVR)

In [11]:
# Scale target for SVR
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# Train SVR
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
svr_model.fit(X_train_encoded, y_train_scaled)

# Predictions (inverse transform)
svr_train_pred_scaled = svr_model.predict(X_train_encoded)
svr_test_pred_scaled = svr_model.predict(X_test_encoded)

svr_train_pred = scaler_y.inverse_transform(svr_train_pred_scaled.reshape(-1, 1)).ravel()
svr_test_pred = scaler_y.inverse_transform(svr_test_pred_scaled.reshape(-1, 1)).ravel()

# Metrics
svr_train_rmse = np.sqrt(mean_squared_error(y_train, svr_train_pred))
svr_test_rmse = np.sqrt(mean_squared_error(y_test, svr_test_pred))
svr_train_mae = mean_absolute_error(y_train, svr_train_pred)
svr_test_mae = mean_absolute_error(y_test, svr_test_pred)
svr_train_r2 = r2_score(y_train, svr_train_pred)
svr_test_r2 = r2_score(y_test, svr_test_pred)

print('=== SVR ===')
print(f'Train RMSE: {svr_train_rmse:.4f}')
print(f'Test RMSE:  {svr_test_rmse:.4f}')
print(f'Train MAE:  {svr_train_mae:.4f}')
print(f'Test MAE:   {svr_test_mae:.4f}')
print(f'Train R²:   {svr_train_r2:.4f}')
print(f'Test R²:    {svr_test_r2:.4f}')

=== SVR ===
Train RMSE: 1.7484
Test RMSE:  1.0764
Train MAE:  0.3693
Test MAE:   0.4944
Train R²:   0.2244
Test R²:    0.2504


## 4. XGBoost

In [12]:
# Train XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_encoded, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train_encoded)
xgb_test_pred = xgb_model.predict(X_test_encoded)

# Metrics
xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))
xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)
xgb_train_r2 = r2_score(y_train, xgb_train_pred)
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print('=== XGBoost ===')
print(f'Train RMSE: {xgb_train_rmse:.4f}')
print(f'Test RMSE:  {xgb_test_rmse:.4f}')
print(f'Train MAE:  {xgb_train_mae:.4f}')
print(f'Test MAE:   {xgb_test_mae:.4f}')
print(f'Train R²:   {xgb_train_r2:.4f}')
print(f'Test R²:    {xgb_test_r2:.4f}')

=== XGBoost ===
Train RMSE: 0.9950
Test RMSE:  2.2959
Train MAE:  0.5152
Test MAE:   0.6223
Train R²:   0.7488
Test R²:    -2.4103


In [13]:
# Cross-validation
xgb_cv_rmse = np.sqrt(-cross_val_score(
    xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                     objective='reg:squarederror', random_state=42, n_jobs=-1),
    X_train_encoded, y_train, cv=cv, scoring='neg_mean_squared_error').mean())
xgb_cv_r2 = cross_val_score(
    xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                     objective='reg:squarederror', random_state=42, n_jobs=-1),
    X_train_encoded, y_train, cv=cv, scoring='r2').mean()

print(f'CV RMSE: {xgb_cv_rmse:.4f}')
print(f'CV R²:   {xgb_cv_r2:.4f}')

CV RMSE: 2.2969
CV R²:   -0.5659


In [14]:
# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(len(xgb_importance)), xgb_importance['Importance'])
plt.yticks(range(len(xgb_importance)), xgb_importance['Feature'])
plt.xlabel('Importance')
plt.title('XGBoost - Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/xgb_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## Model Comparison

In [15]:
# Compile results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'SVR', 'XGBoost'],
    'Train_RMSE': [lr_train_rmse, rf_train_rmse, svr_train_rmse, xgb_train_rmse],
    'Test_RMSE': [lr_test_rmse, rf_test_rmse, svr_test_rmse, xgb_test_rmse],
    'Train_MAE': [lr_train_mae, rf_train_mae, svr_train_mae, xgb_train_mae],
    'Test_MAE': [lr_test_mae, rf_test_mae, svr_test_mae, xgb_test_mae],
    'Train_R2': [lr_train_r2, rf_train_r2, svr_train_r2, xgb_train_r2],
    'Test_R2': [lr_test_r2, rf_test_r2, svr_test_r2, xgb_test_r2],
    'CV_RMSE': [lr_cv_rmse, rf_cv_rmse, np.nan, xgb_cv_rmse],
    'CV_R2': [lr_cv_r2, rf_cv_r2, np.nan, xgb_cv_r2]
})

print('=== MODEL COMPARISON ===')
print(results.to_string(index=False))

=== MODEL COMPARISON ===
            Model  Train_RMSE  Test_RMSE  Train_MAE  Test_MAE  Train_R2   Test_R2  CV_RMSE     CV_R2
Linear Regression    1.619829   1.129645   0.493948  0.668563  0.334284  0.174422 1.893073 -0.006439
    Random Forest    0.850005   1.413934   0.255945  0.517733  0.816687 -0.293398 2.042859 -0.195552
              SVR    1.748414   1.076391   0.369315  0.494403  0.224398  0.250427      NaN       NaN
          XGBoost    0.994950   2.295923   0.515244  0.622331  0.748839 -2.410272 2.296877 -0.565935


In [16]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['Test_RMSE', 'Test_MAE', 'Test_R2']
titles = ['Test RMSE (lower is better)', 'Test MAE (lower is better)', 'Test R² (higher is better)']
colors = ['coral', 'skyblue', 'mediumseagreen']

for i, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[i]
    bars = ax.bar(results['Model'], results[metric], color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title)
    ax.set_ylabel(metric.replace('_', ' '))
    ax.tick_params(axis='x', rotation=15)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [17]:
# Save results
results.to_csv('../outputs/reports/model_comparison.csv', index=False)
print('Results saved to outputs/reports/model_comparison.csv')

Results saved to outputs/reports/model_comparison.csv


## Predicted vs Actual Plots

In [18]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
models = [
    ('Linear Regression', lr_test_pred),
    ('Random Forest', rf_test_pred),
    ('SVR', svr_test_pred),
    ('XGBoost', xgb_test_pred)
]

for i, (name, preds) in enumerate(models):
    ax = axes[i // 2, i % 2]
    ax.scatter(y_test, preds, alpha=0.5, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    ax.set_xlabel('Actual Global Sales')
    ax.set_ylabel('Predicted Global Sales')
    ax.set_title(f'{name}: Predicted vs Actual')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

## Residual Analysis

In [19]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (name, preds) in enumerate(models):
    ax = axes[i // 2, i % 2]
    residuals = y_test - preds
    ax.scatter(preds, residuals, alpha=0.5, s=10)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('Predicted Global Sales')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{name}: Residual Plot')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/residual_plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [20]:
# Export held-out predictions for independent metric verification.
verification = pd.DataFrame({
    'CSV_row_index': y_test.index,
    'Actual_Global_Sales': y_test.to_numpy(),
    'Training_mean_baseline': y_train.mean(),
    'Linear Regression': lr_test_pred,
    'Random Forest': rf_test_pred,
    'SVR': svr_test_pred,
    'XGBoost': xgb_test_pred,
})
verification.to_csv('../outputs/reports/heldout_predictions.csv', index=False)
